# Guidance and Control

**CFG (Classifier-Free Guidance):** Run the model twice (conditional + unconditional), then extrapolate in the conditional direction. Higher guidance = more prompt adherence but less diversity.

**ControlNet:** Inject spatial structure (edges, depth, pose) into the generation process via zero-initialized convolutions. The key trick: zero init means ControlNet starts as identity and gradually learns to inject control signal during training.

**Time:** ~2 hrs. Needs ~6GB VRAM.

In [ ]:
# GPU: ~6 GB VRAM (FP16)
import torch
from diffusers import (
    StableDiffusionPipeline,
    StableDiffusionControlNetPipeline,
    ControlNetModel,
)
from diffusers.utils import load_image
import matplotlib.pyplot as plt
import numpy as np
import cv2
from PIL import Image
from tqdm.auto import tqdm
import requests
from io import BytesIO

device = "cuda" if torch.cuda.is_available() else "cpu"
pipe = StableDiffusionPipeline.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-v1-5",
    torch_dtype=torch.float16,
    safety_checker=None,
).to(device)
print(f"Pipeline loaded on {device}")

In [ ]:
# CFG Sweep: same prompt, same seed, varying guidance_scale
# GPU: ~6 GB VRAM

prompt = "a photo of a golden retriever in a meadow"
seed = 42
guidance_values = [1.0, 3.0, 5.0, 7.5, 10.0, 15.0, 20.0, 30.0]
num_steps = 20

cfg_images = []
for gs in tqdm(guidance_values, desc="CFG sweep"):
    generator = torch.Generator(device=device).manual_seed(seed)
    result = pipe(
        prompt,
        num_inference_steps=num_steps,
        guidance_scale=gs,
        generator=generator,
    )
    cfg_images.append(result.images[0])

# Display as 2x4 grid
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for i, (ax, img, gs) in enumerate(zip(axes.flat, cfg_images, guidance_values)):
    ax.imshow(img)
    ax.set_title(f"CFG = {gs}", fontsize=13)
    ax.axis("off")

plt.suptitle(f"Classifier-Free Guidance Sweep | '{prompt}' | seed={seed}", fontsize=15)
plt.tight_layout()
plt.show()

# Observations:
# CFG=1.0: essentially unconditional, ignores prompt
# CFG=3-7.5: sweet spot, good quality + prompt alignment
# CFG=15+: oversaturated, artifacts, but very prompt-adherent
# CFG=30: extreme artifacts and color saturation

In [ ]:
# Steps Sweep: same prompt, same seed, varying inference steps
# GPU: ~6 GB VRAM

step_values = [1, 2, 4, 8, 15, 25, 50]
guidance_scale = 7.5

step_images = []
for steps in tqdm(step_values, desc="Steps sweep"):
    generator = torch.Generator(device=device).manual_seed(seed)
    result = pipe(
        prompt,
        num_inference_steps=steps,
        guidance_scale=guidance_scale,
        generator=generator,
    )
    step_images.append(result.images[0])

# Display
fig, axes = plt.subplots(1, len(step_values), figsize=(4 * len(step_values), 4))
for ax, img, steps in zip(axes, step_images, step_values):
    ax.imshow(img)
    ax.set_title(f"{steps} steps", fontsize=12)
    ax.axis("off")

plt.suptitle(f"Inference Steps Sweep | CFG={guidance_scale} | seed={seed}", fontsize=14)
plt.tight_layout()
plt.show()

# 1-2 steps: mostly noise
# 4-8: coarse structure emerges
# 15-25: good quality, diminishing returns beyond
# 50: marginal improvement over 25

In [ ]:
# Negative prompts demonstration

prompt_neg_demo = "a portrait of a woman in a garden"

generator = torch.Generator(device=device).manual_seed(seed)
img_no_neg = pipe(
    prompt_neg_demo,
    num_inference_steps=25,
    guidance_scale=7.5,
    generator=generator,
).images[0]

generator = torch.Generator(device=device).manual_seed(seed)
img_quality_neg = pipe(
    prompt_neg_demo,
    negative_prompt="blurry, low quality, distorted",
    num_inference_steps=25,
    guidance_scale=7.5,
    generator=generator,
).images[0]

generator = torch.Generator(device=device).manual_seed(seed)
img_style_neg = pipe(
    prompt_neg_demo,
    negative_prompt="photograph, realistic, photorealistic",
    num_inference_steps=25,
    guidance_scale=7.5,
    generator=generator,
).images[0]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(img_no_neg)
axes[0].set_title("No negative prompt")
axes[0].axis("off")

axes[1].imshow(img_quality_neg)
axes[1].set_title('Negative: "blurry, low quality, distorted"')
axes[1].axis("off")

axes[2].imshow(img_style_neg)
axes[2].set_title('Negative: "photograph, realistic"')
axes[2].axis("off")

plt.suptitle(f"Negative Prompt Effect | '{prompt_neg_demo}'", fontsize=14)
plt.tight_layout()
plt.show()

# How it works: the negative prompt replaces the empty string in the unconditional
# prediction. CFG then extrapolates AWAY from the negative prompt content.
# noise_pred = neg_pred + guidance * (cond_pred - neg_pred)
# This pushes the output away from the negative prompt's direction in embedding space.

In [ ]:
# ControlNet with Canny edges
# GPU: ~6 GB VRAM

# Load ControlNet model
controlnet = ControlNetModel.from_pretrained(
    "lllyasviel/sd-controlnet-canny",
    torch_dtype=torch.float16,
)
pipe_cn = StableDiffusionControlNetPipeline.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-v1-5",
    controlnet=controlnet,
    torch_dtype=torch.float16,
    safety_checker=None,
).to(device)
print("ControlNet pipeline loaded")

# Load or create a source image
try:
    img_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/a/a7/Camponotus_flavomarginatus_ant.jpg/320px-Camponotus_flavomarginatus_ant.jpg"
    response = requests.get(img_url, timeout=10)
    source_img = Image.open(BytesIO(response.content)).convert("RGB")
except Exception:
    # Fallback: generate an image with the base pipeline
    print("URL failed, generating source image with base SD")
    generator = torch.Generator(device=device).manual_seed(99)
    source_img = pipe(
        "a simple house with a tree, children's drawing",
        num_inference_steps=20,
        guidance_scale=7.5,
        generator=generator,
    ).images[0]

source_img = source_img.resize((512, 512))

# Extract Canny edges
source_np = np.array(source_img)
edges = cv2.Canny(source_np, 100, 200)
edges_3ch = np.stack([edges] * 3, axis=-1)  # ControlNet expects 3-channel
canny_image = Image.fromarray(edges_3ch)

# Generate with ControlNet
cn_prompt = "a highly detailed oil painting, masterpiece, vibrant colors"
generator = torch.Generator(device=device).manual_seed(42)
cn_result = pipe_cn(
    cn_prompt,
    image=canny_image,
    num_inference_steps=30,
    guidance_scale=7.5,
    generator=generator,
).images[0]

# Display: source | canny edges | ControlNet output
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(source_img)
axes[0].set_title("Source Image")
axes[0].axis("off")

axes[1].imshow(edges, cmap="gray")
axes[1].set_title("Canny Edges")
axes[1].axis("off")

axes[2].imshow(cn_result)
axes[2].set_title(f"ControlNet: '{cn_prompt[:40]}...'")
axes[2].axis("off")

plt.suptitle("ControlNet with Canny Edge Conditioning", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Side-by-side: same prompt WITH and WITHOUT ControlNet conditioning

comparison_prompt = "a futuristic cyberpunk building, neon lights, detailed architecture"

# Without ControlNet (base SD)
generator = torch.Generator(device=device).manual_seed(42)
img_no_control = pipe(
    comparison_prompt,
    num_inference_steps=30,
    guidance_scale=7.5,
    generator=generator,
).images[0]

# With ControlNet (same seed, same prompt, Canny conditioning)
generator = torch.Generator(device=device).manual_seed(42)
img_with_control = pipe_cn(
    comparison_prompt,
    image=canny_image,
    num_inference_steps=30,
    guidance_scale=7.5,
    generator=generator,
).images[0]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(canny_image)
axes[0].set_title("Canny Edge Condition")
axes[0].axis("off")

axes[1].imshow(img_no_control)
axes[1].set_title("Without ControlNet")
axes[1].axis("off")

axes[2].imshow(img_with_control)
axes[2].set_title("With ControlNet")
axes[2].axis("off")

plt.suptitle(f"ControlNet Structure Preservation | '{comparison_prompt[:50]}...'", fontsize=13)
plt.tight_layout()
plt.show()

# Notice: ControlNet output follows the edge structure of the source image,
# while the base SD output has completely different spatial layout.

In [ ]:
# Inspect ControlNet zero-conv weights
#
# ControlNet copies the encoder of the UNet and adds zero-initialized
# convolution layers ("zero convs") at each connection point. At init,
# these output zeros, meaning ControlNet has NO effect on the base model.
# During training, the zero convs gradually learn to inject spatial signal.

print("ControlNet Architecture Overview:")
print(f"Total parameters: {sum(p.numel() for p in controlnet.parameters()):,}")
print()

# Find zero-conv layers
zero_conv_layers = []
for name, module in controlnet.named_modules():
    if "controlnet_down_blocks" in name or "controlnet_mid_block" in name:
        if isinstance(module, torch.nn.Conv2d):
            zero_conv_layers.append((name, module))

print(f"Found {len(zero_conv_layers)} zero-conv layers:\n")

# Note: after training, these are no longer zero -- they've learned to inject signal
means = []
stds = []
layer_names = []

for name, layer in zero_conv_layers:
    w = layer.weight.data.float()
    w_mean = w.mean().item()
    w_std = w.std().item()
    w_max = w.abs().max().item()
    means.append(w_mean)
    stds.append(w_std)
    short_name = name.split(".")[-2] + "." + name.split(".")[-1] if "." in name else name
    layer_names.append(short_name)
    print(f"  {name}")
    print(f"    shape: {list(w.shape)} | mean: {w_mean:.6f} | std: {w_std:.6f} | max|w|: {w_max:.6f}")

# Also check the dedicated zero-conv layers in controlnet_down_blocks and controlnet_mid_block
print("\n--- Controlnet output convolutions ---")
for name, param in controlnet.named_parameters():
    if "controlnet_down_blocks" in name and "weight" in name:
        w = param.data.float()
        print(f"  {name}: mean={w.mean():.6f}, std={w.std():.6f}, max|w|={w.abs().max():.6f}")
    if "controlnet_mid_block" in name and "weight" in name:
        w = param.data.float()
        print(f"  {name}: mean={w.mean():.6f}, std={w.std():.6f}, max|w|={w.abs().max():.6f}")

# These weights are small (trained model has learned subtle control signals).
# At initialization (before training), these would all be exactly 0.0.
# Zero initialization is the key insight: the pretrained SD UNet is unaffected
# at the start of training, so gradients flow cleanly and training is stable.

# Visualize weight statistics
if means:
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6))
    x = range(len(means))
    ax1.bar(x, means, color="steelblue", alpha=0.7)
    ax1.set_ylabel("Mean weight")
    ax1.set_title("Zero-Conv Layer Weight Statistics (Trained Model)")
    ax1.axhline(y=0, color="red", linestyle="--", alpha=0.5)
    ax1.set_xticks([])

    ax2.bar(x, stds, color="coral", alpha=0.7)
    ax2.set_ylabel("Std of weights")
    ax2.set_xlabel("Zero-conv layer index")
    plt.tight_layout()
    plt.show()

## Checkpoint

**Why does higher quality take longer?**

1. **CFG doubles the forward pass** -- we run the UNet twice per step (conditional + unconditional prediction)
2. **More denoising steps = more forward passes** -- each step refines the image, but with diminishing returns
3. **The fundamental tradeoff:** compute vs quality

At inference:
```
total_cost = num_steps x 2 (for CFG) x model_forward_pass
```

ControlNet adds ~35% compute (parallel encoder that mirrors UNet's downsampling path).

**Further reading:**
- Classifier-Free Guidance: [arXiv:2207.12598](https://arxiv.org/abs/2207.12598)
- ControlNet: [arXiv:2302.05543](https://arxiv.org/abs/2302.05543)